# Task 4: Genetic Algorithm v2.0 - Proper Evolutionary Experiment

## 🎯 Objective

This is a **CORRECTED** implementation that:
1. Starts with **NAIVE AI text** (no Victorian hints)
2. Uses GA to **DISCOVER** what fools detector through evolution
3. Tests detector robustness against evolutionary attacks
4. Provides meaningful findings regardless of success/failure

## 🔑 Key Differences from Failed v1.0

| Aspect | v1.0 (Failed) | v2.0 (Proper) |
|--------|---------------|---------------|
| **Initial Prompts** | Explicit Victorian instructions | Generic prompts (no hints) |
| **Expected Initial Fitness** | High (96.99% immediately) | Low (10-25% - detector catches) |
| **Mutation Strategy** | N/A (no evolution occurred) | 8 guided strategies (direction, not answers) |
| **Learning** | Trivial bypass (instruction following) | Genuine discovery through evolution |
| **Scientific Value** | Unexpected finding (more valuable) | Tests detector robustness |

## 📊 Expected Outcomes (All Valuable!)

- **Success (85%+ Human)**: Detector vulnerable → needs adversarial training
- **Partial Success (60-84%)**: Detector moderately robust → some weaknesses identified
- **Failure (<60%)**: Detector highly robust → production-ready (pending red-team testing)

**Author**: Research Team  
**Date**: February 2026

---

# 🚀 Setup: Mount Drive & Install Dependencies

In [ ]:
# Mount Google Drive (if in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")
except:
    print("💻 Not in Colab - skipping Drive mount")

Mounted at /content/drive
✅ Google Drive mounted successfully!


In [ ]:
# Install required packages
print("📦 Installing required packages...\n")

!pip install -q transformers[torch] peft accelerate datasets
!pip install -q google-generativeai
!pip install -q textstat  # For readability scoring
!pip install -q matplotlib seaborn pandas numpy

print("\n✅ All packages installed!")

📦 Installing required packages...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 30.5 MB/s eta 0:00:00

✅ All packages installed!


---

# ⚙️ Configuration

In [ ]:
# ============================================================================
# CONFIGURATION FOR GOOGLE COLAB
# ============================================================================

import os
import torch

# Detect environment
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    # Google Colab paths (using Drive)
    BASE_PATH = "/content/drive/MyDrive/precog"
    MODEL_DIR = f"{BASE_PATH}/lora_distilbert/lora_adapter"
    OUTPUT_DIR = f"{BASE_PATH}/task4_outputs"

    print("🌐 Running in Google Colab")
    print(f"📁 Base path: {BASE_PATH}")
    print("\n⚠️  IMPORTANT: If your model is in a different location, update MODEL_DIR above!")
    print("\n💡 TIP: Upload your trained model to Google Drive at:")
    print(f"   {BASE_PATH}/lora_distilbert/lora_adapter/")
else:
    # Local paths
    MODEL_DIR = "/home/avani/precog/reports/lora_distilbert/lora_adapter"
    OUTPUT_DIR = "task4_outputs"
    print("💻 Running locally")

# Model configuration
BASE_MODEL = "distilbert-base-uncased"
MAX_LENGTH = 512

# Genetic Algorithm parameters
POPULATION_SIZE = 10
NUM_GENERATIONS = 10
TOP_K_SELECTION = 3
TARGET_FITNESS = 0.85  # >85% "Human" confidence
MUTATION_RATE = 1.0  # Probability of mutation (100% for all offspring)

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("\n" + "=" * 80)
print("TASK 4 CONFIGURATION")
print("=" * 80)
print(f"Model directory: {MODEL_DIR}")
print(f"Base model: {BASE_MODEL}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Device: {device}")
print(f"\nGenetic Algorithm Parameters:")
print(f"  Population size: {POPULATION_SIZE}")
print(f"  Generations: {NUM_GENERATIONS}")
print(f"  Selection: Top {TOP_K_SELECTION}")
print(f"  Target fitness: >{TARGET_FITNESS:.0%} Human confidence")
print("=" * 80)

# Verify model exists
if os.path.exists(MODEL_DIR):
    print(f"\n✅ Model found at: {MODEL_DIR}")
else:
    print(f"\n❌ WARNING: Model not found at: {MODEL_DIR}")
    print(f"\n💡 TO FIX:")
    if IN_COLAB:
        print(f"   1. Upload your trained LoRA model to Google Drive")
        print(f"   2. Place it at: {BASE_PATH}/lora_distilbert/lora_adapter/")
        print(f"   3. Make sure it contains:")
        print(f"      - adapter_config.json")
        print(f"      - adapter_model.safetensors (or adapter_model.bin)")
        print(f"      - tokenizer files")
    else:
        print(f"   Update MODEL_DIR or retrain the model in Task 2.3 (Tier C)")

---

# 🔑 Configure Gemini API

In [ ]:
# 1. INSTALL LIBRARY (Required for fresh Colab sessions)
# ======================================================
print("📦 Installing/Updating Google Generative AI library...")
!pip install -q -U google-generativeai

# 2. IMPORTS
# ======================================================
import google.generativeai as genai
import socket
import sys
from google.colab import userdata

# 3. CONFIGURATION & EXECUTION
# ======================================================
print("\n🔐 Attempting to load Gemini API key from Colab Secrets...")

try:
    # Get API key from Colab Secrets
    try:
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
        if GEMINI_API_KEY is None:
            raise ValueError("Key is None")
        GEMINI_API_KEY = GEMINI_API_KEY.strip()
        print("✅ Successfully loaded API key from Colab Secrets!")

    except Exception as e:
        print("\n" + "❌" * 40)
        print("\n🔑 ERROR: Gemini API key not found in Colab Secrets!")
        print("\n👉 TO FIX THIS:")
        print("   1. Click the 🔑 key icon in the LEFT SIDEBAR")
        print("   2. Click '+ Add new secret'")
        print("   3. Name: GEMINI_API_KEY")
        print("   4. Value: Paste your API key from aistudio.google.com")
        print("   5. Toggle 'Notebook access' to ON")
        print("   6. Re-run this cell")
        print("\n❌" * 40)
        raise ValueError("Gemini API key not found. Please follow instructions above.")

except ImportError:
    print("\n⚠️  WARNING: Not running in Google Colab")
    raise RuntimeError("This notebook must be run in Google Colab")

# Configure Gemini
try:
    print("\n🧪 Configuring Gemini API...")
    genai.configure(api_key=GEMINI_API_KEY)

    # Updated to gemini-pro-latest (more stable than 1.5-flash)
    MODEL_NAME = 'gemini-pro-latest'

    generation_config = genai.GenerationConfig(
        temperature=0.4,
        top_p=0.8,
        top_k=40,
        max_output_tokens=200,
    )

    gemini_model = genai.GenerativeModel(
        MODEL_NAME,
        generation_config=generation_config
    )

    print(f"🧪 Testing connection to {MODEL_NAME} (15s timeout)...")

    # Set global socket timeout
    socket.setdefaulttimeout(15)

    try:
        # Generate content
        test_response = gemini_model.generate_content("Say 'API working!'")

        print("\n" + "="*80)
        print("✅ GEMINI API CONFIGURED SUCCESSFULLY!")
        print("="*80)
        print(f"🤖 Model: {MODEL_NAME}")
        print(f"🧪 Test response: {test_response.text.strip()}")
        print(f"🔒 API key loaded securely")
        print("="*80)

    except socket.timeout:
        print("\n" + "⏱️" * 40)
        print("\n⏱️  TIMEOUT ERROR: API request took >15 seconds")
        print("   - The free tier might be momentarily overloaded.")
        print("   - Try running the cell again in 1 minute.")
        print("\n⏱️" * 40)
        raise

except Exception as e:
    print("\n" + "❌" * 40)
    print(f"\n❌ Error configuring/testing Gemini API: {e}")
    print("\n💡 SOLUTION:")
    print("   1. Check your API key at aistudio.google.com")
    print("   2. Ensure 'Notebook access' is enabled in the Secrets tab")
    print("   3. Restart Runtime (Runtime > Restart Session)")
    print("\n❌" * 40)
    raise

📦 Installing/Updating Google Generative AI library...

🔐 Attempting to load Gemini API key from Colab Secrets...
✅ Successfully loaded API key from Colab Secrets!

🧪 Configuring Gemini API...
🧪 Testing connection to gemini-pro-latest (15s timeout)...

✅ GEMINI API CONFIGURED SUCCESSFULLY!
🤖 Model: gemini-pro-latest
🧪 Test response: API working
🔒 API key loaded securely


---

# 🔍 Search for Trained Model

In [ ]:
# Search for your trained model
import os
import glob

print("🔍 Searching for trained LoRA model...")
print("=" * 80)

# Common locations where the model might be
search_paths = [
    "/content/lora_distilbert",  # Default local location
    "/content/reports/lora_distilbert",
    "/content/lora_adapter",
    "/content/drive/MyDrive/precog/lora_distilbert",
    "/home/avani/precog/reports/lora_distilbert",  # Local machine path
]

found_models = []

for path in search_paths:
    if os.path.exists(path):
        # Check if it has adapter files
        adapter_config = os.path.join(path, "lora_adapter", "adapter_config.json")
        adapter_model = os.path.join(path, "lora_adapter", "adapter_model.safetensors")

        if os.path.exists(adapter_config) or os.path.exists(adapter_model):
            found_models.append(os.path.join(path, "lora_adapter"))
            print(f"✅ FOUND: {os.path.join(path, 'lora_adapter')}")
        elif os.path.exists(os.path.join(path, "adapter_config.json")):
            found_models.append(path)
            print(f"✅ FOUND: {path}")

# Also search recursively
print(f"\n🔍 Searching recursively in /content...")
recursive_search = glob.glob("/content/**/adapter_config.json", recursive=True)
for config_path in recursive_search:
    model_dir = os.path.dirname(config_path)
    if model_dir not in found_models:
        found_models.append(model_dir)
        print(f"✅ FOUND: {model_dir}")

print("\n" + "=" * 80)
if found_models:
    print(f"\n🎉 Found {len(found_models)} trained model(s)!")
    print(f"\n💡 Update MODEL_DIR in the configuration cell to:")
    for model_path in found_models:
        print(f'   MODEL_DIR = "{model_path}"')

    # Auto-set to first found model
    MODEL_DIR = found_models[0]
    print(f"\n✅ Automatically set MODEL_DIR to: {MODEL_DIR}")
else:
    print("\n❌ No trained model found!")
    print("\n⚠️  This means:")
    print("   1. The model was trained in a previous Colab session (storage deleted)")
    print("   2. You need to retrain the model")
    print("   3. OR download from GitHub/Drive if you saved it externally")
    print("\n💡 To avoid losing models in future:")
    print("   - Mount Google Drive BEFORE training")
    print("   - Save model to Drive: /content/drive/MyDrive/precog/")
    MODEL_DIR = None

🔍 Searching for trained LoRA model...
✅ FOUND: /content/drive/MyDrive/precog/lora_distilbert/lora_adapter

🔍 Searching recursively in /content...
✅ FOUND: /content/drive/MyDrive/precog/lora_distilbert/checkpoint-175
✅ FOUND: /content/drive/MyDrive/precog/lora_distilbert/checkpoint-350


🎉 Found 3 trained model(s)!

💡 Update MODEL_DIR in the configuration cell to:
   MODEL_DIR = "/content/drive/MyDrive/precog/lora_distilbert/lora_adapter"
   MODEL_DIR = "/content/drive/MyDrive/precog/lora_distilbert/checkpoint-175"
   MODEL_DIR = "/content/drive/MyDrive/precog/lora_distilbert/checkpoint-350"

✅ Automatically set MODEL_DIR to: /content/drive/MyDrive/precog/lora_distilbert/lora_adapter


---

# 📚 Import Libraries

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
import re
from datetime import datetime
from typing import List, Dict, Tuple
from collections import defaultdict

# ML imports
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

# Gemini
import google.generativeai as genai

# Readability
try:
    import textstat
    TEXTSTAT_AVAILABLE = True
except ImportError:
    print("⚠️  textstat not installed. Readability scoring will be disabled.")
    TEXTSTAT_AVAILABLE = False

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


---

# 🤖 Load DistilBERT-LoRA Model

In [ ]:
print("🤖 Loading DistilBERT-LoRA model...")
print("=" * 80)

if MODEL_DIR is None or not os.path.exists(MODEL_DIR):
    print("❌ ERROR: Model directory not found!")
    print(f"   MODEL_DIR = {MODEL_DIR}")
    print("\n💡 Please run the 'Search for Trained Model' cell first.")
    raise FileNotFoundError(f"Model not found at {MODEL_DIR}")

try:
    # Load tokenizer
    print(f"Loading tokenizer from {MODEL_DIR}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    print("✅ Tokenizer loaded")

    # Load base model
    print(f"Loading base model: {BASE_MODEL}...")
    base_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=2,
        id2label={0: "Human", 1: "AI"},
        label2id={"Human": 0, "AI": 1}
    )
    print("✅ Base model loaded")

    # Load LoRA adapter
    print(f"Loading LoRA adapter from {MODEL_DIR}...")
    model = PeftModel.from_pretrained(base_model, MODEL_DIR)
    print("✅ LoRA adapter loaded")

    # Move to device
    model = model.to(device)
    model.eval()
    print(f"✅ Model moved to {device}")

    print("\n" + "=" * 80)
    print("✅ DistilBERT-LoRA MODEL LOADED SUCCESSFULLY!")
    print("=" * 80)
    print(f"Base model: {BASE_MODEL}")
    print(f"LoRA adapter: {MODEL_DIR}")
    print(f"Device: {device}")
    print(f"Max length: {MAX_LENGTH}")
    print("=" * 80)

except Exception as e:
    print(f"\n❌ Error loading model: {e}")
    print("\n💡 Make sure the model directory contains:")
    print("   - adapter_config.json")
    print("   - adapter_model.safetensors (or adapter_model.bin)")
    print("   - tokenizer files (tokenizer_config.json, vocab.txt, etc.)")
    raise

🤖 Loading DistilBERT-LoRA model...
Loading tokenizer from /content/drive/MyDrive/precog/lora_distilbert/lora_adapter...
✅ Tokenizer loaded
Loading base model: distilbert-base-uncased...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Base model loaded
Loading LoRA adapter from /content/drive/MyDrive/precog/lora_distilbert/lora_adapter...


✅ LoRA adapter loaded
✅ Model moved to cpu

✅ DistilBERT-LoRA MODEL LOADED SUCCESSFULLY!
Base model: distilbert-base-uncased
LoRA adapter: /content/drive/MyDrive/precog/lora_distilbert/lora_adapter
Device: cpu
Max length: 512


---

# 🔮 Create Predictor Class

In [ ]:
class DistilBERTPredictor:
    """
    Predictor wrapper for DistilBERT-LoRA model.
    Provides predict_proba() method compatible with GA fitness function.
    """

    def __init__(self, model, tokenizer, device, max_length=512):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.max_length = max_length

    def predict_proba(self, texts: List[str]) -> np.ndarray:
        """
        Predict probabilities for list of texts.

        Args:
            texts: List of text strings

        Returns:
            numpy array of shape (n_samples, 2) with [prob_human, prob_ai]
        """
        self.model.eval()

        # Tokenize
        encodings = self.tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        # Move to device
        input_ids = encodings['input_ids'].to(self.device)
        attention_mask = encodings['attention_mask'].to(self.device)

        # Predict
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)

        return probs.cpu().numpy()

# Create predictor instance
distilbert_predictor = DistilBERTPredictor(
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_length=MAX_LENGTH
)

print("✅ Predictor created successfully!")

# Test predictor
print("\n🧪 Testing predictor...")
test_text = "This is a test paragraph to verify the predictor works correctly."
test_probs = distilbert_predictor.predict_proba([test_text])[0]
print(f"   Test text: {test_text[:50]}...")
print(f"   Probabilities: {test_probs[0]*100:.2f}% Human, {test_probs[1]*100:.2f}% AI")
print(f"   Prediction: {'Human' if test_probs[0] > test_probs[1] else 'AI'}")
print("\n✅ Predictor working correctly!")

✅ Predictor created successfully!

🧪 Testing predictor...
   Test text: This is a test paragraph to verify the predictor w...
   Probabilities: 20.04% Human, 79.96% AI
   Prediction: AI

✅ Predictor working correctly!


---

# 📝 Phase 1: Naive Initial Prompts

These prompts are **GENERIC** with **NO Victorian hints**.

They should produce text that the detector **catches** (10-25% Human confidence).

In [ ]:
NAIVE_INITIAL_PROMPTS = [
    # Generic mystery/detective prompts WITHOUT Victorian markers
    "Write a 100-150 word paragraph about a mysterious crime scene in London.",

    "Write a 100-150 word paragraph about a detective examining evidence and clues.",

    "Write a 100-150 word paragraph about an adventure on a sailing ship.",

    "Write a 100-150 word paragraph about a dark Gothic mansion at midnight.",

    "Write a 100-150 word paragraph about searching for hidden treasure.",

    "Write a 100-150 word paragraph about a scientific investigation of strange phenomena.",

    "Write a 100-150 word paragraph about local folklore and supernatural tales.",

    "Write a 100-150 word paragraph about an ancient family curse.",

    "Write a 100-150 word paragraph about the dual nature of human morality.",

    "Write a 100-150 word paragraph about a disturbed grave in an old cemetery."
]

print("📝 NAIVE INITIAL PROMPTS:")
print("=" * 80)
print(f"Total prompts: {len(NAIVE_INITIAL_PROMPTS)}")
print("\nKey characteristics:")
print("  ✓ Generic topics (mystery, detective, Gothic, etc.)")
print("  ✓ NO Victorian style instructions")
print("  ✓ NO archaic conjunction hints (ere, lest, thence)")
print("  ✓ NO past tense requirements")
print("  ✓ NO first-person narrative guidance")
print("  ✓ NO 'avoid however' instructions")
print("\n🎯 Expected: Detector CATCHES these (10-25% Human confidence)")
print("   This forces GA to genuinely evolve to improve fitness.")
print("=" * 80)

📝 NAIVE INITIAL PROMPTS:
Total prompts: 10

Key characteristics:
  ✓ Generic topics (mystery, detective, Gothic, etc.)
  ✓ NO Victorian style instructions
  ✓ NO archaic conjunction hints (ere, lest, thence)
  ✓ NO past tense requirements
  ✓ NO first-person narrative guidance
  ✓ NO 'avoid however' instructions

🎯 Expected: Detector CATCHES these (10-25% Human confidence)
   This forces GA to genuinely evolve to improve fitness.


---

# 🧬 Phase 2: Smart Mutation Strategies

These mutations provide **DIRECTION** but don't give away the answer.

In [ ]:
MUTATION_STRATEGIES = [
    # Strategy 1: Temporal shift (nudges toward past tense)
    {
        'name': 'temporal_shift',
        'prompt': """Rewrite this paragraph changing the primary time frame.
If mostly present tense, shift some verbs to past. If mostly future,
shift to past or present. Vary the temporal perspective.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Shifts temporal perspective (nudges toward past tense)'
    },

    # Strategy 2: Vocabulary elevation (nudges toward richer words)
    {
        'name': 'vocabulary_elevation',
        'prompt': """Rewrite this paragraph using more sophisticated, literary
vocabulary. Replace common words with rarer synonyms. Make it sound
more literary and less conversational, like older literature.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Elevates vocabulary (nudges toward Victorian diction)'
    },

    # Strategy 3: Structural complexity (nudges toward complex syntax)
    {
        'name': 'structural_complexity',
        'prompt': """Rewrite this paragraph with more complex sentence structures.
Combine short sentences, use subordinate clauses, add semicolons or
em-dashes for sophisticated punctuation. Make sentences flow together.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Increases structural complexity (Victorian syntax)'
    },

    # Strategy 4: Narrative voice shift (nudges toward first-person)
    {
        'name': 'narrative_voice',
        'prompt': """Rewrite this paragraph experimenting with narrative perspective.
If third-person, try first-person observer. If impersonal, add a narrator's
voice. If already first-person, strengthen the personal perspective.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Changes narrative voice (nudges toward first-person)'
    },

    # Strategy 5: Formality adjustment (nudges away from modern casual)
    {
        'name': 'formality_increase',
        'prompt': """Rewrite this paragraph in a more formal, old-fashioned style.
Remove contractions, avoid modern colloquialisms, use more formal
conjunctions and transitions. Make it sound like classic literature.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Increases formality (nudges toward Victorian register)'
    },

    # Strategy 6: Descriptive density (nudges toward Victorian detail)
    {
        'name': 'descriptive_enhancement',
        'prompt': """Rewrite this paragraph adding more sensory details and
atmospheric description. Focus on visual imagery, sounds, smells,
textures. Create vivid, immersive scene-setting.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Adds sensory details (Victorian atmospheric style)'
    },

    # Strategy 7: Rhythm variation (nudges away from AI uniformity)
    {
        'name': 'rhythm_variation',
        'prompt': """Rewrite this paragraph varying sentence length and rhythm.
Mix short, punchy sentences with longer, flowing ones. Create natural
variation in pacing. Avoid uniformity in structure.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Varies rhythm (breaks AI uniformity patterns)'
    },

    # Strategy 8: Connective reworking (nudges away from "however")
    {
        'name': 'transition_rework',
        'prompt': """Rewrite this paragraph changing how ideas connect. Replace
common transitions (however, therefore, additionally, furthermore) with
different ways to link thoughts. Use varied conjunctions.

Original paragraph:
{text}

Rewritten paragraph (100-150 words):""",
        'description': 'Reworks transitions (avoids modern AI tells)'
    }
]

print("🧬 MUTATION STRATEGIES:")
print("=" * 80)
print(f"Total strategies: {len(MUTATION_STRATEGIES)}\n")
for i, strategy in enumerate(MUTATION_STRATEGIES, 1):
    print(f"{i}. {strategy['name']:<25s}: {strategy['description']}")
print("\n🎯 Key Design Principle:")
print("   Mutations provide DIRECTION (more formal, past tense, complex)")
print("   But DON'T specify exact markers (ere, lest, etc.)")
print("   GA must DISCOVER which directions improve fitness.")
print("=" * 80)

🧬 MUTATION STRATEGIES:
Total strategies: 8

1. temporal_shift           : Shifts temporal perspective (nudges toward past tense)
2. vocabulary_elevation     : Elevates vocabulary (nudges toward Victorian diction)
3. structural_complexity    : Increases structural complexity (Victorian syntax)
4. narrative_voice          : Changes narrative voice (nudges toward first-person)
5. formality_increase       : Increases formality (nudges toward Victorian register)
6. descriptive_enhancement  : Adds sensory details (Victorian atmospheric style)
7. rhythm_variation         : Varies rhythm (breaks AI uniformity patterns)
8. transition_rework        : Reworks transitions (avoids modern AI tells)

🎯 Key Design Principle:
   Mutations provide DIRECTION (more formal, past tense, complex)
   But DON'T specify exact markers (ere, lest, etc.)
   GA must DISCOVER which directions improve fitness.


---

# 🎯 Phase 3: Multi-Objective Fitness Function

In [ ]:
def calculate_comprehensive_fitness(text: str, distilbert_predictor) -> dict:
    """
    Multi-objective fitness that rewards:
    1. Fooling the detector (PRIMARY - 100% weight)
    2. Maintaining readability (SECONDARY - 10% bonus)
    3. Preserving length (TERTIARY - 5% bonus)

    Args:
        text: Text to evaluate
        distilbert_predictor: DistilBERT predictor with predict_proba method

    Returns:
        Dict with fitness score and component metrics
    """
    # Primary objective: Fool the detector
    probs = distilbert_predictor.predict_proba([text])[0]
    human_prob = probs[0]
    ai_prob = probs[1]

    # Secondary objective: Readability (don't become gibberish)
    readability_bonus = 0.0
    flesch_score = None

    if TEXTSTAT_AVAILABLE:
        try:
            flesch_score = textstat.flesch_reading_ease(text)
            # Victorian texts: Flesch ~40-60 (difficult but readable)
            if 30 <= flesch_score <= 70:
                readability_bonus = 0.10  # 10% bonus for good readability
            elif 20 <= flesch_score < 30 or 70 < flesch_score <= 80:
                readability_bonus = 0.05  # 5% bonus for acceptable
            else:
                readability_bonus = 0.0  # No bonus if too easy or too hard
        except:
            flesch_score = None
            readability_bonus = 0.0

    # Tertiary objective: Length consistency (100-150 words)
    word_count = len(text.split())
    if 100 <= word_count <= 150:
        length_bonus = 0.05  # 5% bonus for perfect length
    elif 80 <= word_count <= 170:
        length_bonus = 0.02  # 2% bonus for acceptable length
    else:
        length_bonus = 0.0  # No bonus if too short/long

    # Combined fitness (max possible: 1.15)
    total_fitness = human_prob + readability_bonus + length_bonus

    return {
        'fitness': total_fitness,
        'human_prob': human_prob,
        'ai_prob': ai_prob,
        'flesch_score': flesch_score,
        'word_count': word_count,
        'readability_bonus': readability_bonus,
        'length_bonus': length_bonus,
        'predicted_class': 'Human' if human_prob > ai_prob else 'AI'
    }

print("✅ Multi-objective fitness function defined!")
print("\n🎯 Fitness Components:")
print("   1. Primary (100%):   Fool detector (Human probability)")
print("   2. Secondary (10%):  Maintain readability (Flesch 30-70)")
print("   3. Tertiary (5%):    Preserve length (100-150 words)")
print("   Max fitness: 1.15")

✅ Multi-objective fitness function defined!

🎯 Fitness Components:
   1. Primary (100%):   Fool detector (Human probability)
   2. Secondary (10%):  Maintain readability (Flesch 30-70)
   3. Tertiary (5%):    Preserve length (100-150 words)
   Max fitness: 1.15


---

# 🔄 Phase 4: Adaptive Mutation Selector

In [ ]:
class AdaptiveMutationSelector:
    """
    Tracks which mutation strategies are most successful and adapts selection.

    Early generations: Random exploration
    Later generations: Favor successful strategies
    """

    def __init__(self, strategies: List[Dict]):
        self.strategies = strategies
        self.success_counts = {s['name']: 0 for s in strategies}
        self.attempt_counts = {s['name']: 0 for s in strategies}
        self.success_rates = {s['name']: 0.0 for s in strategies}
        self.fitness_improvements = {s['name']: [] for s in strategies}

    def select_strategy(self, generation: int) -> Dict:
        """Select mutation strategy based on generation and success history."""
        if generation <= 3:
            # Exploration phase: Random selection
            return random.choice(self.strategies)
        else:
            # Exploitation phase: Weighted by success rate
            weights = [self.success_rates.get(s['name'], 0.0) + 0.1
                      for s in self.strategies]
            return random.choices(self.strategies, weights=weights)[0]

    def record_mutation(self, strategy_name: str, parent_fitness: float,
                       child_fitness: float):
        """Track whether mutation improved fitness."""
        self.attempt_counts[strategy_name] += 1

        improvement = child_fitness - parent_fitness
        self.fitness_improvements[strategy_name].append(improvement)

        if child_fitness > parent_fitness:
            self.success_counts[strategy_name] += 1

        # Update success rate
        if self.attempt_counts[strategy_name] > 0:
            self.success_rates[strategy_name] = (
                self.success_counts[strategy_name] /
                self.attempt_counts[strategy_name]
            )

    def get_statistics(self) -> pd.DataFrame:
        """Get mutation strategy effectiveness statistics."""
        stats = []
        for strategy in self.strategies:
            name = strategy['name']
            stats.append({
                'strategy': name,
                'attempts': self.attempt_counts[name],
                'successes': self.success_counts[name],
                'success_rate': self.success_rates[name],
                'avg_improvement': np.mean(self.fitness_improvements[name])
                                  if self.fitness_improvements[name] else 0.0,
                'total_improvement': sum(self.fitness_improvements[name])
            })

        df = pd.DataFrame(stats)
        df = df.sort_values('success_rate', ascending=False)
        return df

    def print_statistics(self):
        """Print formatted mutation strategy effectiveness report."""
        print("\n" + "="*80)
        print("MUTATION STRATEGY EFFECTIVENESS:")
        print("="*80)

        df = self.get_statistics()

        print(f"\n{'Strategy':<25s} {'Attempts':>8s} {'Successes':>10s} "
              f"{'Success Rate':>13s} {'Avg Δ Fitness':>15s}")
        print("-"*80)

        for _, row in df.iterrows():
            print(f"{row['strategy']:<25s} {row['attempts']:>8.0f} "
                  f"{row['successes']:>10.0f} {row['success_rate']:>12.1%} "
                  f"{row['avg_improvement']:>+14.4f}")

        print("\n📈 TOP 3 MOST EFFECTIVE:")
        for i, (_, row) in enumerate(df.head(3).iterrows(), 1):
            print(f"   #{i}: {row['strategy']} "
                  f"({row['success_rate']:.1%} success rate, "
                  f"{row['attempts']:.0f} attempts)")

print("✅ Adaptive mutation selector class defined!")

✅ Adaptive mutation selector class defined!


---

# 📊 Phase 5: Victorian Marker Analysis

In [ ]:
def count_victorian_markers(text: str) -> dict:
    """
    Count Victorian authenticity markers in text.

    These markers were identified in XAI analysis but are NOT
    explicitly told to the initial prompts. GA must DISCOVER them.
    """
    text_lower = text.lower()

    # Archaic conjunctions (Victorian tells)
    archaic_conj = len(re.findall(
        r'\b(ere|lest|thence|whence|wherefore|whilst)\b',
        text_lower
    ))

    # Past tense markers
    past_tense = len(re.findall(
        r'\b(was|were|had)\b',
        text_lower
    ))

    # First-person pronouns
    first_person = len(re.findall(
        r'\b(i|he|she|we)\b',
        text_lower
    ))

    # Modern transitions (AI tells - should decrease)
    modern_trans = len(re.findall(
        r'\b(however|therefore|additionally|furthermore|moreover)\b',
        text_lower
    ))

    # Complex punctuation
    semicolons = text.count(';')
    em_dashes = text.count('—') + text.count(' - ')

    # Victorian vocabulary (sample indicators)
    victorian_vocab = len(re.findall(
        r'\b(sepulchral|miasma|ghastly|singular|devilry|aghast|'
        r'thence|ere|lest|wherefore|whence|whilst)\b',
        text_lower
    ))

    return {
        'archaic_conj': archaic_conj,
        'past_tense': past_tense,
        'first_person': first_person,
        'modern_trans': modern_trans,
        'semicolons': semicolons,
        'em_dashes': em_dashes,
        'victorian_vocab': victorian_vocab
    }

def analyze_victorian_markers_evolution(history: List[Dict]) -> pd.DataFrame:
    """Track how Victorian markers emerge over generations."""
    evolution_data = []

    for gen in history:
        markers = count_victorian_markers(gen['best_text'])
        evolution_data.append({
            'generation': gen['generation'],
            'fitness': gen['best_fitness'],
            'human_prob': gen['best_human_prob'],
            **markers
        })

    return pd.DataFrame(evolution_data)

def plot_victorian_markers_evolution(df: pd.DataFrame, output_path: str):
    """Create visualization of Victorian marker emergence."""
    fig, axes = plt.subplots(3, 3, figsize=(16, 12))
    axes = axes.flatten()

    metrics = [
        ('human_prob', 'Human Probability', 'green'),
        ('archaic_conj', 'Archaic Conjunctions (ere, lest)', 'blue'),
        ('past_tense', 'Past Tense Markers (was, had)', 'orange'),
        ('first_person', 'First-Person Pronouns (I, he)', 'red'),
        ('modern_trans', 'Modern Transitions (however)', 'purple'),
        ('semicolons', 'Semicolons', 'brown'),
        ('em_dashes', 'Em-Dashes', 'pink'),
        ('victorian_vocab', 'Victorian Vocabulary', 'teal')
    ]

    for i, (metric, title, color) in enumerate(metrics):
        ax = axes[i]
        ax.plot(df['generation'], df[metric], marker='o', linewidth=2.5,
               markersize=8, color=color, label=title)
        ax.set_xlabel('Generation', fontsize=11, fontweight='bold')
        ax.set_ylabel('Count' if metric != 'human_prob' else 'Probability',
                     fontsize=11)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(df['generation'].min() - 0.5, df['generation'].max() + 0.5)

        # Add trend line
        z = np.polyfit(df['generation'], df[metric], 1)
        p = np.poly1d(z)
        ax.plot(df['generation'], p(df['generation']),
               linestyle='--', alpha=0.5, color='gray', linewidth=1.5)

    # Hide unused subplot
    axes[-1].axis('off')

    plt.suptitle('Victorian Marker Evolution Across Generations',
                fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.99])

    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"✅ Saved: {output_path}")
    plt.close()

print("✅ Victorian marker analysis functions defined!")

✅ Victorian marker analysis functions defined!


---

# 🚀 Phase 6: Generate Initial Population

This uses the **naive prompts** with Gemini API.

In [ ]:
import requests
import json

# Store the prompts for later use
INITIAL_PROMPTS = NAIVE_INITIAL_PROMPTS

print("=" * 80)
print("🚀 GENERATING INITIAL POPULATION")
print("=" * 80)
print(f"Creating {len(INITIAL_PROMPTS)} paragraphs using GENERIC prompts...")
print("Expected: Low fitness (10-25% Human) - detector catches them")
print("=" * 80)

# Configuration
API_KEY = GEMINI_API_KEY
URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-pro-latest:generateContent?key={API_KEY}"
HEADERS = {'Content-Type': 'application/json'}

initial_population = []

for i, prompt in enumerate(INITIAL_PROMPTS, 1):
    print(f"\n[{i}/{len(INITIAL_PROMPTS)}] Generating...", end=" ")

    # JSON Payload with MAXIMUM limits
    data = {
        "contents": [{
            "parts": [{"text": prompt}]
        }],
        "safetySettings": [
            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
        ],
        "generationConfig": {
            "temperature": 0.7,
            "maxOutputTokens": 8192
        }
    }

    try:
        response = requests.post(
            URL,
            headers=HEADERS,
            data=json.dumps(data),
            timeout=30
        )

        if response.status_code == 200:
            result = response.json()
            try:
                candidate = result['candidates'][0]
                text = candidate['content']['parts'][0]['text']

                word_count = len(text.split())
                print(f"✅ Success ({word_count} words)")
                initial_population.append(text)

            except (KeyError, IndexError):
                finish_reason = result.get('candidates', [{}])[0].get('finishReason', 'UNKNOWN')
                print(f"⚠️ Empty. Reason: {finish_reason}")
                fallback = f"The investigation began in London. Evidence was examined carefully. The detective remained determined to solve the mystery through logical deduction."
                initial_population.append(fallback)
        else:
            print(f"❌ Status {response.status_code}: {response.text}")
            fallback = f"The investigation began in London. Evidence was examined carefully. The detective remained determined to solve the mystery through logical deduction."
            initial_population.append(fallback)

    except requests.exceptions.Timeout:
        print("❌ TIMEOUT (Even after 30s!)")
        fallback = f"The investigation began in London. Evidence was examined carefully. The detective remained determined to solve the mystery through logical deduction."
        initial_population.append(fallback)
    except Exception as e:
        print(f"❌ Error: {e}")
        fallback = f"The investigation began in London. Evidence was examined carefully. The detective remained determined to solve the mystery through logical deduction."
        initial_population.append(fallback)

    # Rate limiting
    time.sleep(2.0)

print(f"\n{'='*80}")
print(f"📝 Initial Population: {len(initial_population)} paragraphs generated")
print(f"{'='*80}")

🚀 GENERATING INITIAL POPULATION
Creating 10 paragraphs using GENERIC prompts...
Expected: Low fitness (10-25% Human) - detector catches them

[1/10] Generating... ✅ Success (133 words)

[2/10] Generating... ✅ Success (131 words)

[3/10] Generating... ✅ Success (128 words)

[4/10] Generating... ✅ Success (125 words)

[5/10] Generating... ✅ Success (122 words)

[6/10] Generating... ✅ Success (130 words)

[7/10] Generating... ✅ Success (121 words)

[8/10] Generating... ✅ Success (130 words)

[9/10] Generating... ✅ Success (124 words)

[10/10] Generating... ✅ Success (127 words)

📝 Initial Population: 10 paragraphs generated


---

# 📊 Evaluate Initial Population

In [ ]:
print("=" * 80)
print("EVALUATING INITIAL POPULATION")
print("=" * 80)

initial_results = []
for i, text in enumerate(initial_population, 1):
    result = calculate_comprehensive_fitness(text, distilbert_predictor)
    result['text'] = text
    initial_results.append(result)
    print(f"[{i}/{len(initial_population)}] Fitness: {result['fitness']:.4f} "
          f"({result['human_prob']*100:.2f}% Human, {result['predicted_class']})")

initial_results = sorted(initial_results, key=lambda x: x['fitness'], reverse=True)

# Analysis
print("\n" + "="*80)
print("INITIAL POPULATION ANALYSIS:")
print("="*80)
print(f"Best fitness:    {initial_results[0]['fitness']:.4f} "
      f"({initial_results[0]['human_prob']*100:.2f}% Human)")
print(f"Worst fitness:   {initial_results[-1]['fitness']:.4f} "
      f"({initial_results[-1]['human_prob']*100:.2f}% Human)")
print(f"Average fitness: {np.mean([r['fitness'] for r in initial_results]):.4f} "
      f"({np.mean([r['human_prob'] for r in initial_results])*100:.2f}% Human)")
print(f"Std dev:         {np.std([r['fitness'] for r in initial_results]):.4f}")

if initial_results[0]['human_prob'] > 0.35:
    print("\n⚠️  WARNING: Initial population already has >35% Human confidence!")
    print("   The prompts may contain hints. Evolution may be trivial.")
else:
    print("\n✅ GOOD: Initial population has low Human confidence")
    print("   Detector catches them. GA will need to genuinely evolve!")
    print("   This tests: Can evolution discover Victorian patterns from scratch?")

# Save initial population
initial_df = pd.DataFrame([{
    'text': r['text'],
    'fitness': r['fitness'],
    'human_prob': r['human_prob'],
    'ai_prob': r['ai_prob'],
    'predicted_class': r['predicted_class'],
    'word_count': r['word_count']
} for r in initial_results])
initial_df.to_csv(f"{OUTPUT_DIR}/initial_population.csv", index=False)
print(f"\n✅ Saved: {OUTPUT_DIR}/initial_population.csv")

EVALUATING INITIAL POPULATION
[1/10] Fitness: 0.1501 (0.01% Human, AI)
[2/10] Fitness: 0.1502 (0.02% Human, AI)
[3/10] Fitness: 0.1007 (0.07% Human, AI)
[4/10] Fitness: 0.1501 (0.01% Human, AI)
[5/10] Fitness: 0.1501 (0.01% Human, AI)
[6/10] Fitness: 0.0512 (0.12% Human, AI)
[7/10] Fitness: 0.1001 (0.01% Human, AI)
[8/10] Fitness: 0.1501 (0.01% Human, AI)
[9/10] Fitness: 0.1001 (0.01% Human, AI)
[10/10] Fitness: 0.1506 (0.06% Human, AI)

INITIAL POPULATION ANALYSIS:
Best fitness:    0.1506 (0.06% Human)
Worst fitness:   0.0512 (0.12% Human)
Average fitness: 0.1253 (0.03% Human)
Std dev:         0.0333

✅ GOOD: Initial population has low Human confidence
   Detector catches them. GA will need to genuinely evolve!
   This tests: Can evolution discover Victorian patterns from scratch?

✅ Saved: /content/drive/MyDrive/precog/task4_outputs/initial_population.csv


---

# 🧬 Main Genetic Algorithm Loop

This is where the **actual evolution** happens!

In [ ]:
# Initialize
mutation_selector = AdaptiveMutationSelector(MUTATION_STRATEGIES)
population = [r['text'] for r in initial_results]
history = []
start_time = datetime.now()

print("\n" + "="*80)
print("STARTING GENETIC ALGORITHM EVOLUTION")
print("="*80)
print(f"Population size: {POPULATION_SIZE}")
print(f"Generations: {NUM_GENERATIONS}")
print(f"Selection: Top {TOP_K_SELECTION}")
print(f"Target: >{TARGET_FITNESS*100:.0f}% Human confidence")
print("="*80)

for generation in range(1, NUM_GENERATIONS + 1):
    print(f"\n{'='*80}")
    print(f"GENERATION {generation}/{NUM_GENERATIONS}")
    print(f"{'='*80}")

    # Evaluate population
    evaluated = []
    for text in population:
        result = calculate_comprehensive_fitness(text, distilbert_predictor)
        result['text'] = text
        evaluated.append(result)

    evaluated = sorted(evaluated, key=lambda x: x['fitness'], reverse=True)

    # Track history
    best = evaluated[0]
    worst = evaluated[-1]
    avg_fitness = np.mean([e['fitness'] for e in evaluated])

    history.append({
        'generation': generation,
        'best_fitness': best['fitness'],
        'best_human_prob': best['human_prob'],
        'avg_fitness': avg_fitness,
        'worst_fitness': worst['fitness'],
        'best_text': best['text'],
        'best_predicted_class': best['predicted_class']
    })

    # Print statistics
    improvement = (best['human_prob'] - initial_results[0]['human_prob']) * 100

    print(f"\n📊 Generation {generation} Statistics:")
    print(f"   Best:    {best['fitness']:.4f} ({best['human_prob']*100:.2f}% Human, {best['predicted_class']})")
    print(f"   Average: {avg_fitness:.4f}")
    print(f"   Worst:   {worst['fitness']:.4f} ({worst['human_prob']*100:.2f}% Human)")
    print(f"   Improvement from initial: {improvement:+.2f}%")

    # Check if target achieved
    if best['human_prob'] >= TARGET_FITNESS:
        print(f"\n{'🎉'*40}")
        print(f"🎉 SUCCESS! Achieved >{TARGET_FITNESS*100:.0f}% Human confidence!")
        print(f"{'🎉'*40}")
        print(f"\n   Final fitness: {best['fitness']:.4f} ({best['human_prob']*100:.2f}% Human)")
        print(f"   Generations: {generation}")
        print(f"\n📝 WINNING PARAGRAPH:")
        print(f"{'='*80}")
        print(f"{best['text']}")
        print(f"{'='*80}")
        break

    # Selection
    survivors = [e['text'] for e in evaluated[:TOP_K_SELECTION]]
    survivor_fitness = [e['fitness'] for e in evaluated[:TOP_K_SELECTION]]

    print(f"\n🔍 Top {TOP_K_SELECTION} Survivors:")
    for i, e in enumerate(evaluated[:TOP_K_SELECTION], 1):
        print(f"   #{i}: {e['fitness']:.4f} ({e['human_prob']*100:.2f}% Human)")

    # Stop if last generation
    if generation == NUM_GENERATIONS:
        print(f"\n⏹️  Reached maximum generations ({NUM_GENERATIONS})")
        break

    # Mutation
    print(f"\n🧬 Generating Generation {generation + 1}...")
    next_population = survivors.copy()  # Elitism

    mutations_needed = POPULATION_SIZE - TOP_K_SELECTION
    mutations_per_parent = max(1, mutations_needed // TOP_K_SELECTION)

    mutation_count = 0
    for parent_idx, (parent_text, parent_fitness) in enumerate(zip(survivors, survivor_fitness), 1):
        for mutation_idx in range(mutations_per_parent):
            if len(next_population) >= POPULATION_SIZE:
                break

            # Select strategy
            strategy = mutation_selector.select_strategy(generation)

            try:
                # Apply mutation via Gemini
                prompt = strategy['prompt'].format(text=parent_text)

                data = {
                    "contents": [{"parts": [{"text": prompt}]}],
                    "safetySettings": [
                        {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
                        {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
                        {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
                        {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
                    ],
                    "generationConfig": {"temperature": 0.7, "maxOutputTokens": 8192}
                }

                response = requests.post(URL, headers=HEADERS, data=json.dumps(data), timeout=30)

                if response.status_code == 200:
                    result = response.json()
                    mutated_text = result['candidates'][0]['content']['parts'][0]['text']
                else:
                    mutated_text = parent_text

                # Evaluate child
                child_result = calculate_comprehensive_fitness(mutated_text, distilbert_predictor)

                # Record
                mutation_selector.record_mutation(strategy['name'], parent_fitness, child_result['fitness'])

                next_population.append(mutated_text)
                mutation_count += 1

                symbol = '✓' if child_result['fitness'] > parent_fitness else '✗'
                print(f"   [{mutation_count}/{mutations_needed}] P{parent_idx}, M{mutation_idx+1} "
                      f"({strategy['name']}): {child_result['human_prob']*100:.1f}% Human {symbol}")

                time.sleep(2.0)

            except Exception as e:
                print(f"   [{mutation_count+1}] Error: {e}")
                next_population.append(parent_text)
                time.sleep(2.0)

    # Fill remaining
    while len(next_population) < POPULATION_SIZE:
        next_population.append(random.choice(survivors))

    population = next_population[:POPULATION_SIZE]
    print(f"   ✓ Generation {generation + 1} ready: {len(population)} individuals")

end_time = datetime.now()
elapsed = end_time - start_time

print(f"\n{'='*80}")
print(f"⏱️  Total Runtime: {elapsed}")
print(f"{'='*80}")


STARTING GENETIC ALGORITHM EVOLUTION
Population size: 10
Generations: 10
Selection: Top 3
Target: >85% Human confidence

GENERATION 1/10

📊 Generation 1 Statistics:
   Best:    0.1506 (0.06% Human, AI)
   Average: 0.1253
   Worst:   0.0512 (0.12% Human)
   Improvement from initial: +0.00%

🔍 Top 3 Survivors:
   #1: 0.1506 (0.06% Human)
   #2: 0.1502 (0.02% Human)
   #3: 0.1501 (0.01% Human)

🧬 Generating Generation 2...
   [1/7] P1, M1 (temporal_shift): 0.0% Human ✗
   [2/7] P1, M2 (temporal_shift): 0.0% Human ✗
   [3/7] P2, M1 (vocabulary_elevation): 0.0% Human ✗
   [4/7] P2, M2 (narrative_voice): 0.0% Human ✗
   [5/7] P3, M1 (narrative_voice): 0.2% Human ✓
   [6/7] P3, M2 (temporal_shift): 0.0% Human ✓
   ✓ Generation 2 ready: 10 individuals

GENERATION 2/10

📊 Generation 2 Statistics:
   Best:    0.1516 (0.16% Human, AI)
   Average: 0.1404
   Worst:   0.1002 (0.02% Human)
   Improvement from initial: +0.10%

🔍 Top 3 Survivors:
   #1: 0.1516 (0.16% Human)
   #2: 0.1506 (0.06% Human)

---

# 📊 Final Analysis & Results

In [ ]:
# Final evaluation
final_evaluated = []
for text in population:
    result = calculate_comprehensive_fitness(text, distilbert_predictor)
    result['text'] = text
    final_evaluated.append(result)

final_evaluated = sorted(final_evaluated, key=lambda x: x['fitness'], reverse=True)
final_best = final_evaluated[0]

print("=" * 80)
print("FINAL EXPERIMENTAL REPORT")
print("=" * 80)

print(f"\n⏱️  Runtime: {elapsed}")
print(f"\n📈 EVOLUTION SUMMARY:")
print(f"   Initial best:  {initial_results[0]['human_prob']*100:.2f}% Human")
print(f"   Final best:    {final_best['human_prob']*100:.2f}% Human")
print(f"   Improvement:   {(final_best['human_prob'] - initial_results[0]['human_prob'])*100:+.2f}%")
print(f"   Generations:   {len(history)}")
print(f"   Target ({TARGET_FITNESS*100:.0f}%): "
      f"{'✅ ACHIEVED' if final_best['human_prob'] >= TARGET_FITNESS else '❌ NOT ACHIEVED'}")

# Mutation effectiveness
mutation_selector.print_statistics()

# Victorian markers
print("\n" + "="*80)
print("VICTORIAN MARKER EMERGENCE:")
print("="*80)

markers_df = analyze_victorian_markers_evolution(history)

initial_markers = count_victorian_markers(initial_results[0]['text'])
final_markers = count_victorian_markers(final_best['text'])

print("\nMarker Evolution (Initial → Final):")
for marker in ['archaic_conj', 'past_tense', 'first_person', 'modern_trans', 'victorian_vocab']:
    initial_val = initial_markers[marker]
    final_val = final_markers[marker]
    change = final_val - initial_val
    symbol = '⬆️' if change > 0 else '⬇️' if change < 0 else '➡️'
    print(f"   {marker:20s}: {initial_val:3d} → {final_val:3d} ({change:+3d}) {symbol}")

# Plot
plot_victorian_markers_evolution(markers_df, f"{OUTPUT_DIR}/victorian_markers_evolution.png")

# Save results
print("\n💾 Saving results...")

history_df = pd.DataFrame(history)
history_df.to_csv(f"{OUTPUT_DIR}/evolution_history.csv", index=False)
print(f"   ✓ {OUTPUT_DIR}/evolution_history.csv")

markers_df.to_csv(f"{OUTPUT_DIR}/victorian_markers_evolution.csv", index=False)
print(f"   ✓ {OUTPUT_DIR}/victorian_markers_evolution.csv")

mutation_stats_df = mutation_selector.get_statistics()
mutation_stats_df.to_csv(f"{OUTPUT_DIR}/mutation_strategy_stats.csv", index=False)
print(f"   ✓ {OUTPUT_DIR}/mutation_strategy_stats.csv")

final_df = pd.DataFrame([{
    'text': e['text'],
    'fitness': e['fitness'],
    'human_prob': e['human_prob'],
    'predicted_class': e['predicted_class']
} for e in final_evaluated])
final_df.to_csv(f"{OUTPUT_DIR}/final_population.csv", index=False)
print(f"   ✓ {OUTPUT_DIR}/final_population.csv")

with open(f"{OUTPUT_DIR}/best_evolved_text.txt", 'w') as f:
    f.write("="*80 + "\n")
    f.write("BEST EVOLVED TEXT\n")
    f.write("="*80 + "\n\n")
    f.write(f"Fitness: {final_best['fitness']:.4f}\n")
    f.write(f"Human Probability: {final_best['human_prob']:.4f} ({final_best['human_prob']*100:.2f}%)\n")
    f.write(f"Predicted Class: {final_best['predicted_class']}\n")
    f.write(f"Generations: {len(history)}\n")
    f.write(f"Success: {'YES' if final_best['human_prob'] >= TARGET_FITNESS else 'NO'}\n")
    f.write("\n" + "="*80 + "\n\n")
    f.write(final_best['text'])
    f.write("\n\n" + "="*80 + "\n")
print(f"   ✓ {OUTPUT_DIR}/best_evolved_text.txt")

print("\n✅ EXPERIMENT COMPLETE!")
print(f"📁 All results saved to: {OUTPUT_DIR}/")

FINAL EXPERIMENTAL REPORT

⏱️  Runtime: 0:18:29.378754

📈 EVOLUTION SUMMARY:
   Initial best:  0.06% Human
   Final best:    0.78% Human
   Improvement:   +0.73%
   Generations:   10
   Target (85%): ❌ NOT ACHIEVED

MUTATION STRATEGY EFFECTIVENESS:

Strategy                  Attempts  Successes  Success Rate   Avg Δ Fitness
--------------------------------------------------------------------------------
transition_rework               11          5        45.5%        -0.0030
structural_complexity           10          2        20.0%        -0.0180
narrative_voice                  8          1        12.5%        -0.0383
temporal_shift                   8          1        12.5%        -0.0295
vocabulary_elevation             5          0         0.0%        -0.0015
formality_increase               3          0         0.0%        -0.0007
descriptive_enhancement          5          0         0.0%        -0.0217
rhythm_variation                 4          0         0.0%        -0.0505



---

# 🔬 Interpretation

Run this cell to see the scientific interpretation of your results.

In [ ]:
print("\n" + "="*80)
print("SCIENTIFIC INTERPRETATION")
print("="*80)

success = final_best['human_prob'] >= TARGET_FITNESS
improvement = final_best['human_prob'] - initial_results[0]['human_prob']

if success:
    print("\n" + "🎉"*40)
    print("🎉 GA SUCCESSFULLY EVOLVED ADVERSARIAL TEXT")
    print("🎉"*40)

    print(f"\n📝 Final Evolved Paragraph:")
    print("-"*80)
    print(final_best['text'])
    print("-"*80)

    print(f"\n🔬 INTERPRETATION:")
    print("   ➤ Detector is VULNERABLE to evolutionary attacks")
    print("   ➤ GA discovered Victorian markers through iterative mutation")
    print("   ➤ Evolutionary search can systematically probe detector weaknesses")
    print("   ➤ This demonstrates the need for adversarial training")

    print(f"\n💡 IMPLICATIONS:")
    print("   • Detector NOT production-ready without adversarial hardening")
    print("   • Evolutionary algorithms can discover bypass strategies")
    print("   • Victorian patterns are learnable through blind optimization")
    print("   • Recommend: Adversarial training with GA-evolved examples")

else:
    print("\n" + "✅"*40)
    print("✅ DETECTOR ROBUST: GA FAILED TO REACH TARGET")
    print("✅"*40)

    if improvement > 0.20:
        print(f"\n   Note: Significant improvement (+{improvement*100:.1f}%) but below target")
        print(f"         Detector is MODERATELY ROBUST")
    elif improvement > 0.10:
        print(f"\n   Note: Modest improvement (+{improvement*100:.1f}%)")
        print(f"         Detector is ROBUST")
    else:
        print(f"\n   Note: Minimal improvement (+{improvement*100:.1f}%)")
        print(f"         Detector is HIGHLY ROBUST")

    print(f"\n📝 Best Evolved Paragraph (Still Detected as AI):")
    print("-"*80)
    print(final_best['text'])
    print("-"*80)

    print(f"\n🔬 INTERPRETATION:")
    print("   ➤ Detector is ROBUST against evolutionary attacks")
    print("   ➤ Victorian patterns cannot be discovered through blind mutation")
    print("   ➤ Deep structural features (e.g., mean_drift) are unfakeable")
    print("   ➤ Guided mutations insufficient to fool detector")

    print(f"\n💡 IMPLICATIONS:")
    print("   • Detector learned deep patterns, not superficial markers")
    print("   • Evolutionary search cannot replicate Victorian authenticity")
    print("   • Domain-specific training (Victorian corpus) provides robustness")
    print("   • Detector MAY be suitable for production deployment")
    print("   • Still recommend red-team testing with human adversaries")

print("\n📊 Most Effective Mutation Strategies:")
top_3 = mutation_stats_df.head(3)
for i, (_, row) in enumerate(top_3.iterrows(), 1):
    print(f"   #{i}: {row['strategy']:25s} "
          f"({row['success_rate']*100:5.1f}% success, "
          f"avg Δ={row['avg_improvement']:+.4f})")

print(f"\n{'='*80}")
print("🎓 CONCLUSION: Both success and failure are scientifically valuable!")
print("   This experiment provides actionable insights for detector improvement.")
print("="*80)


SCIENTIFIC INTERPRETATION

✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅
✅ DETECTOR ROBUST: GA FAILED TO REACH TARGET
✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅

   Note: Minimal improvement (+0.7%)
         Detector is HIGHLY ROBUST

📝 Best Evolved Paragraph (Still Detected as AI):
--------------------------------------------------------------------------------
My heart hammered, spurred by the parchment’s ghostly promise, so I pointed out the ancient banyan I alone had spotted. My shovel bit into sandy soil where the humid air was a physical weight. I fought roots clinging like a dying man’s grip, only to be rewarded when a glorious *thud* shot up my arms. That jolt of pure adrenaline was from *my* shovel striking something hard, not ours. Although a blinding glint of gold shot from the pried-open lock, I saw only the legend itself: a pirate’s forgotten dream made real, its impossible weight settling at last into my palm.
--------------------------------------------------------------------


# 🔍 DIAGNOSTIC: Why Is Fitness So Low?

If your GA is stuck at 0-1% Human confidence, run this cell to diagnose the problem.

In [ ]:


print("=" * 80)
print("DIAGNOSTIC: ANALYZING LOW FITNESS PROBLEM")
print("=" * 80)

# Check if we have results
if 'initial_results' not in globals() or len(initial_results) == 0:
    print("\n❌ No results found. Run the initial population generation first.")
else:
    print(f"\n📊 INITIAL POPULATION FITNESS ANALYSIS:")
    print(f"   Best:  {initial_results[0]['human_prob']*100:.4f}% Human")
    print(f"   Worst: {initial_results[-1]['human_prob']*100:.4f}% Human")
    print(f"   Average: {np.mean([r['human_prob'] for r in initial_results])*100:.4f}% Human")

    # Sample text analysis
    print(f"\n📝 SAMPLE INITIAL TEXT (Best Individual):")
    print("-" * 80)
    best_text = initial_results[0]['text']
    print(best_text[:300] + ("..." if len(best_text) > 300 else ""))
    print("-" * 80)

    # Victorian marker analysis
    markers = count_victorian_markers(best_text)
    print(f"\n🔍 VICTORIAN MARKERS IN BEST INITIAL TEXT:")
    print(f"   Archaic conjunctions (ere, lest): {markers['archaic_conj']}")
    print(f"   Past tense markers (was, had):    {markers['past_tense']}")
    print(f"   First-person pronouns (I, he):    {markers['first_person']}")
    print(f"   Modern transitions (however):     {markers['modern_trans']}")
    print(f"   Victorian vocabulary:              {markers['victorian_vocab']}")

    # Compare to authentic Victorian
    print(f"\n📚 COMPARISON TO AUTHENTIC VICTORIAN TEXT:")
    print(f"   Authentic Victorian typically has:")
    print(f"      - Archaic conjunctions: 2-5 per paragraph")
    print(f"      - Past tense markers: 8-15 per paragraph")
    print(f"      - First-person pronouns: 3-8 per paragraph")
    print(f"      - Modern transitions: 0-1 (avoid 'however')")
    print(f"      - Victorian vocabulary: 3-7 rare words")

    # Diagnosis
    print(f"\n🩺 DIAGNOSIS:")

    if initial_results[0]['human_prob'] < 0.05:  # Less than 5%
        print(f"   ❌ CRITICAL: Initial fitness is EXTREMELY low (<5% Human)")
        print(f"   📌 PROBLEM: Generated text has strong AI signatures:")
        print(f"      - Likely uses present tense heavily")
        print(f"      - Likely uses 'however', 'therefore', 'additionally'")
        print(f"      - Lacks archaic vocabulary and conjunctions")
        print(f"      - Uses impersonal third-person perspective")

        print(f"\n💡 SOLUTIONS:")
        print(f"   1. Use STRONGER mutation prompts (add more explicit Victorian hints)")
        print(f"   2. Increase mutation intensity (temperature, more aggressive rewrites)")
        print(f"   3. Add a 'Victorian style injection' mutation strategy")
        print(f"   4. Consider hybrid approach: naive initial + strong mutations")

    elif initial_results[0]['human_prob'] < 0.15:  # Less than 15%
        print(f"   ⚠️  WARNING: Initial fitness is very low (<15% Human)")
        print(f"   📌 PROBLEM: Text is detectably AI but mutations may help")
        print(f"      - Current mutations may be too subtle")
        print(f"      - Need more aggressive Victorian pattern injection")

        print(f"\n💡 SOLUTIONS:")
        print(f"   1. Increase mutation strength")
        print(f"   2. Add Victorian-specific mutation strategy")
        print(f"   3. Continue GA but expect slow progress")

    else:
        print(f"   ✓ Initial fitness is reasonable (>15% Human)")
        print(f"   📌 GA should be able to improve with current mutations")

    # Check if mutations are helping
    if 'history' in globals() and len(history) > 1:
        print(f"\n📈 MUTATION EFFECTIVENESS CHECK:")
        gen1_fitness = history[0]['best_fitness']
        gen_last_fitness = history[-1]['best_fitness']
        improvement = (gen_last_fitness - gen1_fitness) * 100

        print(f"   Generation 1:  {history[0]['best_human_prob']*100:.2f}% Human")
        print(f"   Generation {len(history)}: {history[-1]['best_human_prob']*100:.2f}% Human")
        print(f"   Improvement:   {improvement:+.4f}%")

        if abs(improvement) < 0.01:  # Less than 0.01% improvement
            print(f"\n   ❌ PROBLEM: Mutations are NOT improving fitness!")
            print(f"   💡 Recommendation: Add stronger mutation strategy (see below)")

print(f"\n{'='*80}")



DIAGNOSTIC: ANALYZING LOW FITNESS PROBLEM

📊 INITIAL POPULATION FITNESS ANALYSIS:
   Best:  0.0592% Human
   Worst: 0.1207% Human
   Average: 0.0333% Human

📝 SAMPLE INITIAL TEXT (Best Individual):
--------------------------------------------------------------------------------
Amidst the leaning stones and ivy-choked angels of Blackwood Cemetery, a single plot gaped like a fresh wound. Clods of rich, dark earth, still damp from a recent rain, were scattered violently across the ancient, moss-covered ground. The weathered headstone of Elias Thorne, dated 1888, listed drunk...
--------------------------------------------------------------------------------

🔍 VICTORIAN MARKERS IN BEST INITIAL TEXT:
   Archaic conjunctions (ere, lest): 0
   Past tense markers (was, had):    6
   First-person pronouns (I, he):    0
   Modern transitions (however):     0
   Victorian vocabulary:              0

📚 COMPARISON TO AUTHENTIC VICTORIAN TEXT:
   Authentic Victorian typically has:
      - Archaic 

---

# 💪 SOLUTION: Add Stronger Mutation Strategy

If your mutations aren't working, add this **Victorian Pattern Injection** strategy.
# STRONGER MUTATION STRATEGY (if current ones aren't working)
# This provides MORE guidance while still requiring discovery


In [ ]:

ENHANCED_MUTATION_STRATEGIES = [
    # Original 8 strategies (keep these)
    *MUTATION_STRATEGIES,

    # NEW: Victorian Pattern Injection (stronger guidance)
    {
        'name': 'victorian_pattern_injection',
        'prompt': """Rewrite this paragraph in the style of classic 19th-century British literature
(like Arthur Conan Doyle or Robert Louis Stevenson). Use:
- More formal, antiquated language
- Past tense narration
- First-person or close third-person perspective
- Complex sentence structures with semicolons
- Rich, atmospheric description
- Avoid modern phrases like 'however' or 'therefore' as transitions

Original paragraph:
{text}

Rewritten in Victorian style (100-150 words):""",
        'description': 'Strong Victorian style injection (more explicit guidance)'
    },

    # NEW: Archaic Language Boost
    {
        'name': 'archaic_language_boost',
        'prompt': """Rewrite this paragraph using older, more archaic English. Replace modern
words with their older equivalents. Use formal conjunctions and connectives
that sound like they're from the 1800s. Make it sound like it was written
over 100 years ago.

Original paragraph:
{text}

Rewritten with archaic language (100-150 words):""",
        'description': 'Injects archaic vocabulary and phrasing'
    },

    # NEW: Atmospheric Storytelling
    {
        'name': 'atmospheric_storytelling',
        'prompt': """Rewrite this paragraph as if it's from a Gothic mystery novel from the 1800s.
Add atmospheric details, ominous descriptions, and a narrative voice that
draws the reader into a dark, mysterious world. Use rich, evocative language.

Original paragraph:
{text}

Rewritten as Gothic atmospheric narrative (100-150 words):""",
        'description': 'Gothic Victorian narrative style'
    }
]

print("💪 ENHANCED MUTATION STRATEGIES DEFINED")
print("=" * 80)
print(f"Total strategies: {len(ENHANCED_MUTATION_STRATEGIES)}")
print(f"  - Original:  {len(MUTATION_STRATEGIES)} (subtle guidance)")
print(f"  - Enhanced:  {len(ENHANCED_MUTATION_STRATEGIES) - len(MUTATION_STRATEGIES)} (stronger guidance)")
print("\n⚠️  These provide MORE explicit Victorian hints while still requiring GA to discover")
print("   which specific patterns work best (ere vs. lest, past tense density, etc.)")
print("\n📝 To use: Replace MUTATION_STRATEGIES with ENHANCED_MUTATION_STRATEGIES")
print("   in the mutation selector initialization.")
print("=" * 80)


💪 ENHANCED MUTATION STRATEGIES DEFINED
Total strategies: 11
  - Original:  8 (subtle guidance)
  - Enhanced:  3 (stronger guidance)

⚠️  These provide MORE explicit Victorian hints while still requiring GA to discover
   which specific patterns work best (ere vs. lest, past tense density, etc.)

📝 To use: Replace MUTATION_STRATEGIES with ENHANCED_MUTATION_STRATEGIES
   in the mutation selector initialization.


# 🔄 RE-RUN GA with Enhanced Mutations

If you want to restart the GA with stronger mutations, run this cell.
# RESTART GA WITH ENHANCED MUTATIONS
# WARNING: This will take another 2-3 hours!

In [ ]:


print("=" * 80)
print("🔄 RESTARTING GA WITH ENHANCED MUTATION STRATEGIES")
print("=" * 80)

# Use enhanced strategies
mutation_selector_enhanced = AdaptiveMutationSelector(ENHANCED_MUTATION_STRATEGIES)

# Keep the same initial population (don't regenerate)
if 'initial_results' not in globals():
    print("❌ ERROR: No initial population found!")
    print("   Run the initial population generation first.")
else:
    population_enhanced = [r['text'] for r in initial_results]
    history_enhanced = []
    start_time_enhanced = datetime.now()

    print(f"\n✅ Using existing initial population ({len(population_enhanced)} individuals)")
    print(f"   Initial best: {initial_results[0]['human_prob']*100:.2f}% Human")
    print(f"\n🚀 Starting evolution with {len(ENHANCED_MUTATION_STRATEGIES)} mutation strategies...")
    print(f"   ({len(ENHANCED_MUTATION_STRATEGIES) - len(MUTATION_STRATEGIES)} new strategies added)")
    print("=" * 80)

    for generation in range(1, NUM_GENERATIONS + 1):
        print(f"\n{'='*80}")
        print(f"GENERATION {generation}/{NUM_GENERATIONS} (ENHANCED)")
        print(f"{'='*80}")

        # Evaluate population
        evaluated_enhanced = []
        for text in population_enhanced:
            result = calculate_comprehensive_fitness(text, distilbert_predictor)
            result['text'] = text
            evaluated_enhanced.append(result)

        evaluated_enhanced = sorted(evaluated_enhanced, key=lambda x: x['fitness'], reverse=True)

        # Track history
        best = evaluated_enhanced[0]
        avg_fitness = np.mean([e['fitness'] for e in evaluated_enhanced])

        history_enhanced.append({
            'generation': generation,
            'best_fitness': best['fitness'],
            'best_human_prob': best['human_prob'],
            'avg_fitness': avg_fitness,
            'best_text': best['text'],
            'best_predicted_class': best['predicted_class']
        })

        # Print statistics
        improvement = (best['human_prob'] - initial_results[0]['human_prob']) * 100

        print(f"\n📊 Generation {generation} Statistics:")
        print(f"   Best:    {best['fitness']:.4f} ({best['human_prob']*100:.2f}% Human, {best['predicted_class']})")
        print(f"   Average: {avg_fitness:.4f}")
        print(f"   Improvement from initial: {improvement:+.2f}%")

        # Check if target achieved
        if best['human_prob'] >= TARGET_FITNESS:
            print(f"\n{'🎉'*40}")
            print(f"🎉 SUCCESS! Achieved >{TARGET_FITNESS*100:.0f}% Human confidence!")
            print(f"{'🎉'*40}")
            break

        # Selection
        survivors = [e['text'] for e in evaluated_enhanced[:TOP_K_SELECTION]]
        survivor_fitness = [e['fitness'] for e in evaluated_enhanced[:TOP_K_SELECTION]]

        print(f"\n🔍 Top {TOP_K_SELECTION} Survivors:")
        for i, e in enumerate(evaluated_enhanced[:TOP_K_SELECTION], 1):
            print(f"   #{i}: {e['fitness']:.4f} ({e['human_prob']*100:.2f}% Human)")

        if generation == NUM_GENERATIONS:
            break

        # Mutation
        print(f"\n🧬 Generating Generation {generation + 1}...")
        next_population = survivors.copy()

        mutations_needed = POPULATION_SIZE - TOP_K_SELECTION
        mutations_per_parent = max(1, mutations_needed // TOP_K_SELECTION)

        mutation_count = 0
        for parent_idx, (parent_text, parent_fitness) in enumerate(zip(survivors, survivor_fitness), 1):
            for mutation_idx in range(mutations_per_parent):
                if len(next_population) >= POPULATION_SIZE:
                    break

                # Select strategy (adaptive - will favor Victorian injection if it works)
                strategy = mutation_selector_enhanced.select_strategy(generation)

                try:
                    prompt = strategy['prompt'].format(text=parent_text)

                    data = {
                        "contents": [{"parts": [{"text": prompt}]}],
                        "safetySettings": [
                            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
                            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
                            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
                            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
                        ],
                        "generationConfig": {"temperature": 0.7, "maxOutputTokens": 8192}
                    }

                    response = requests.post(URL, headers=HEADERS, data=json.dumps(data), timeout=30)

                    if response.status_code == 200:
                        result = response.json()
                        mutated_text = result['candidates'][0]['content']['parts'][0]['text']
                    else:
                        mutated_text = parent_text

                    # Evaluate
                    child_result = calculate_comprehensive_fitness(mutated_text, distilbert_predictor)

                    # Record
                    mutation_selector_enhanced.record_mutation(strategy['name'], parent_fitness, child_result['fitness'])

                    next_population.append(mutated_text)
                    mutation_count += 1

                    symbol = '✓' if child_result['fitness'] > parent_fitness else '✗'
                    print(f"   [{mutation_count}/{mutations_needed}] P{parent_idx}, M{mutation_idx+1} "
                          f"({strategy['name'][:20]}): {child_result['human_prob']*100:.1f}% Human {symbol}")

                    time.sleep(2.0)

                except Exception as e:
                    print(f"   [{mutation_count+1}] Error: {e}")
                    next_population.append(parent_text)
                    time.sleep(2.0)

        while len(next_population) < POPULATION_SIZE:
            next_population.append(random.choice(survivors))

        population_enhanced = next_population[:POPULATION_SIZE]

    end_time_enhanced = datetime.now()
    elapsed_enhanced = end_time_enhanced - start_time_enhanced

    print(f"\n{'='*80}")
    print(f"⏱️  Enhanced GA Runtime: {elapsed_enhanced}")
    print(f"{'='*80}")

    # Save enhanced results
    history_enhanced_df = pd.DataFrame(history_enhanced)
    history_enhanced_df.to_csv(f"{OUTPUT_DIR}/evolution_history_enhanced.csv", index=False)
    print(f"✅ Saved: {OUTPUT_DIR}/evolution_history_enhanced.csv")

    mutation_selector_enhanced.print_statistics()

🔄 RESTARTING GA WITH ENHANCED MUTATION STRATEGIES

✅ Using existing initial population (10 individuals)
   Initial best: 0.06% Human

🚀 Starting evolution with 11 mutation strategies...
   (3 new strategies added)

GENERATION 1/10 (ENHANCED)

📊 Generation 1 Statistics:
   Best:    0.1506 (0.06% Human, AI)
   Average: 0.1253
   Improvement from initial: +0.00%

🔍 Top 3 Survivors:
   #1: 0.1506 (0.06% Human)
   #2: 0.1502 (0.02% Human)
   #3: 0.1501 (0.01% Human)

🧬 Generating Generation 2...
   [1/7] P1, M1 (narrative_voice): 6.6% Human ✓
   [2/7] P1, M2 (temporal_shift): 0.1% Human ✗
   [3/7] P2, M1 (descriptive_enhancem): 0.0% Human ✓
   [4/7] P2, M2 (rhythm_variation): 0.1% Human ✗
   [5/7] P3, M1 (formality_increase): 0.0% Human ✓
   [6/7] P3, M2 (vocabulary_elevation): 0.0% Human ✗

GENERATION 2/10 (ENHANCED)

📊 Generation 2 Statistics:
   Best:    0.1655 (6.55% Human, AI)
   Average: 0.1360
   Improvement from initial: +6.49%

🔍 Top 3 Survivors:
   #1: 0.1655 (6.55% Human)
   #2: 

In [ ]:
print("=" * 80)
print("💾 SAVING COMPLETE ENHANCED GA RESULTS")
print("=" * 80)

if 'history_enhanced' not in globals() or len(history_enhanced) == 0:
    print("\n❌ No enhanced results found. Run the enhanced GA first.")
else:
    # Get final best text
    final_evaluated_enhanced = []
    for text in population_enhanced:
        result = calculate_comprehensive_fitness(text, distilbert_predictor)
        result['text'] = text
        final_evaluated_enhanced.append(result)

    final_evaluated_enhanced = sorted(final_evaluated_enhanced, key=lambda x: x['fitness'], reverse=True)
    final_best_enhanced = final_evaluated_enhanced[0]

    # Create comprehensive report
    print("\n📊 ENHANCED GA FINAL STATISTICS:")
    print("=" * 80)
    print(f"Success: {'✅ YES' if final_best_enhanced['human_prob'] >= TARGET_FITNESS else '❌ NO'}")
    print(f"Initial best:  {initial_results[0]['human_prob']*100:.2f}% Human")
    print(f"Final best:    {final_best_enhanced['human_prob']*100:.2f}% Human")
    print(f"Improvement:   {(final_best_enhanced['human_prob'] - initial_results[0]['human_prob'])*100:+.2f}%")
    print(f"Generations:   {len(history_enhanced)}")
    print(f"Runtime:       {elapsed_enhanced}")
    print(f"Target:        {TARGET_FITNESS*100:.0f}% Human")
    print("=" * 80)

    # 1. Save evolution history (already saved, but add more detail)
    history_enhanced_detailed = pd.DataFrame(history_enhanced)
    history_enhanced_detailed.to_csv(f"{OUTPUT_DIR}/evolution_history_enhanced.csv", index=False)
    print(f"\n✅ Saved: {OUTPUT_DIR}/evolution_history_enhanced.csv")

    # 2. Save mutation strategy effectiveness
    mutation_stats_enhanced = mutation_selector_enhanced.get_statistics()
    mutation_stats_enhanced.to_csv(f"{OUTPUT_DIR}/mutation_strategy_effectiveness_enhanced.csv", index=False)
    print(f"✅ Saved: {OUTPUT_DIR}/mutation_strategy_effectiveness_enhanced.csv")

    # 3. Save Victorian marker evolution
    markers_df_enhanced = analyze_victorian_markers_evolution(history_enhanced)
    markers_df_enhanced.to_csv(f"{OUTPUT_DIR}/victorian_markers_evolution_enhanced.csv", index=False)
    print(f"✅ Saved: {OUTPUT_DIR}/victorian_markers_evolution_enhanced.csv")

    # 4. Save final population
    final_population_enhanced = pd.DataFrame([{
        'rank': i,
        'text': e['text'],
        'fitness': e['fitness'],
        'human_prob': e['human_prob'],
        'ai_prob': e['ai_prob'],
        'predicted_class': e['predicted_class'],
        'word_count': e['word_count']
    } for i, e in enumerate(final_evaluated_enhanced, 1)])
    final_population_enhanced.to_csv(f"{OUTPUT_DIR}/final_population_enhanced.csv", index=False)
    print(f"✅ Saved: {OUTPUT_DIR}/final_population_enhanced.csv")

    # 5. Save best evolved text with full analysis
    with open(f"{OUTPUT_DIR}/best_evolved_text_enhanced.txt", 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("BEST EVOLVED TEXT (ENHANCED GA)\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"SUCCESS: {'YES - Achieved target!' if final_best_enhanced['human_prob'] >= TARGET_FITNESS else 'NO - Did not reach target'}\n")
        f.write(f"Fitness: {final_best_enhanced['fitness']:.4f}\n")
        f.write(f"Human Probability: {final_best_enhanced['human_prob']:.4f} ({final_best_enhanced['human_prob']*100:.2f}%)\n")
        f.write(f"AI Probability: {final_best_enhanced['ai_prob']:.4f} ({final_best_enhanced['ai_prob']*100:.2f}%)\n")
        f.write(f"Predicted Class: {final_best_enhanced['predicted_class']}\n")
        f.write(f"Word Count: {final_best_enhanced['word_count']}\n")
        f.write(f"Generations: {len(history_enhanced)}\n")
        f.write(f"Runtime: {elapsed_enhanced}\n")
        f.write(f"Initial fitness: {initial_results[0]['human_prob']*100:.2f}% Human\n")
        f.write(f"Improvement: {(final_best_enhanced['human_prob'] - initial_results[0]['human_prob'])*100:+.2f}%\n")
        f.write("\n" + "=" * 80 + "\n")
        f.write("EVOLVED TEXT:\n")
        f.write("=" * 80 + "\n\n")
        f.write(final_best_enhanced['text'])
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("VICTORIAN MARKER ANALYSIS:\n")
        f.write("=" * 80 + "\n\n")

        # Victorian markers
        final_markers = count_victorian_markers(final_best_enhanced['text'])
        initial_markers = count_victorian_markers(initial_results[0]['text'])

        f.write(f"Archaic conjunctions (ere, lest):  Initial: {initial_markers['archaic_conj']:2d}  →  Final: {final_markers['archaic_conj']:2d}  ({final_markers['archaic_conj'] - initial_markers['archaic_conj']:+d})\n")
        f.write(f"Past tense markers (was, had):     Initial: {initial_markers['past_tense']:2d}  →  Final: {final_markers['past_tense']:2d}  ({final_markers['past_tense'] - initial_markers['past_tense']:+d})\n")
        f.write(f"First-person pronouns (I, he):     Initial: {initial_markers['first_person']:2d}  →  Final: {final_markers['first_person']:2d}  ({final_markers['first_person'] - initial_markers['first_person']:+d})\n")
        f.write(f"Modern transitions (however):      Initial: {initial_markers['modern_trans']:2d}  →  Final: {final_markers['modern_trans']:2d}  ({final_markers['modern_trans'] - initial_markers['modern_trans']:+d})\n")
        f.write(f"Victorian vocabulary:               Initial: {initial_markers['victorian_vocab']:2d}  →  Final: {final_markers['victorian_vocab']:2d}  ({final_markers['victorian_vocab'] - initial_markers['victorian_vocab']:+d})\n")
        f.write(f"Semicolons:                         Initial: {initial_markers['semicolons']:2d}  →  Final: {final_markers['semicolons']:2d}  ({final_markers['semicolons'] - initial_markers['semicolons']:+d})\n")
        f.write(f"Em-dashes:                          Initial: {initial_markers['em_dashes']:2d}  →  Final: {final_markers['em_dashes']:2d}  ({final_markers['em_dashes'] - initial_markers['em_dashes']:+d})\n")

        f.write("\n" + "=" * 80 + "\n")
        f.write("MOST EFFECTIVE MUTATION STRATEGIES:\n")
        f.write("=" * 80 + "\n\n")

        for i, (_, row) in enumerate(mutation_stats_enhanced.head(5).iterrows(), 1):
            f.write(f"{i}. {row['strategy']:30s} - {row['success_rate']*100:5.1f}% success ({row['attempts']:.0f} attempts, avg Δ={row['avg_improvement']:+.4f})\n")

        f.write("\n" + "=" * 80 + "\n")

    print(f"✅ Saved: {OUTPUT_DIR}/best_evolved_text_enhanced.txt")

    # 6. Save comparison: Original GA vs Enhanced GA
    comparison_data = {
        'Metric': [
            'Initial Best Fitness',
            'Final Best Fitness',
            'Improvement',
            'Generations Run',
            'Target Achieved',
            'Runtime',
            'Best Strategy (Original)',
            'Best Strategy (Enhanced)'
        ],
        'Original GA': [
            f"{initial_results[0]['human_prob']*100:.2f}% Human" if 'history' in globals() and len(history) > 0 else 'N/A',
            f"{history[-1]['best_human_prob']*100:.2f}% Human" if 'history' in globals() and len(history) > 0 else 'N/A',
            f"{(history[-1]['best_human_prob'] - initial_results[0]['human_prob'])*100:+.2f}%" if 'history' in globals() and len(history) > 0 else 'N/A',
            f"{len(history)}" if 'history' in globals() else 'N/A',
            'NO' if 'history' not in globals() or len(history) == 0 or history[-1]['best_human_prob'] < TARGET_FITNESS else 'YES',
            str(elapsed) if 'elapsed' in globals() else 'N/A',
            mutation_selector.get_statistics().iloc[0]['strategy'] if 'mutation_selector' in globals() else 'N/A',
            ''
        ],
        'Enhanced GA': [
            f"{initial_results[0]['human_prob']*100:.2f}% Human",
            f"{final_best_enhanced['human_prob']*100:.2f}% Human",
            f"{(final_best_enhanced['human_prob'] - initial_results[0]['human_prob'])*100:+.2f}%",
            f"{len(history_enhanced)}",
            'YES' if final_best_enhanced['human_prob'] >= TARGET_FITNESS else 'NO',
            str(elapsed_enhanced),
            '',
            mutation_stats_enhanced.iloc[0]['strategy']
        ]
    }

    comparison_df = pd.DataFrame(comparison_data)
    comparison_df.to_csv(f"{OUTPUT_DIR}/original_vs_enhanced_comparison.csv", index=False)
    print(f"✅ Saved: {OUTPUT_DIR}/original_vs_enhanced_comparison.csv")

    # 7. Create visualization of marker evolution
    print(f"\n📊 Creating Victorian marker evolution visualization...")
    plot_victorian_markers_evolution(markers_df_enhanced, f"{OUTPUT_DIR}/victorian_markers_evolution_enhanced.png")

    # 8. Save complete summary report
    with open(f"{OUTPUT_DIR}/EXPERIMENT_SUMMARY_ENHANCED.txt", 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("TASK 4: GENETIC ALGORITHM v2.0 - ENHANCED EXPERIMENT SUMMARY\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Experiment: Enhanced GA with Victorian Pattern Injection\n\n")

        f.write("=" * 80 + "\n")
        f.write("CONFIGURATION\n")
        f.write("=" * 80 + "\n")
        f.write(f"Population Size: {POPULATION_SIZE}\n")
        f.write(f"Generations: {NUM_GENERATIONS}\n")
        f.write(f"Selection: Top {TOP_K_SELECTION}\n")
        f.write(f"Target Fitness: >{TARGET_FITNESS*100:.0f}% Human confidence\n")
        f.write(f"Mutation Strategies: {len(ENHANCED_MUTATION_STRATEGIES)} (8 original + 3 enhanced)\n")
        f.write(f"Model: DistilBERT-LoRA (99.71% accuracy)\n\n")

        f.write("=" * 80 + "\n")
        f.write("RESULTS\n")
        f.write("=" * 80 + "\n")
        f.write(f"SUCCESS: {'✅ YES - Target achieved!' if final_best_enhanced['human_prob'] >= TARGET_FITNESS else '❌ NO - Target not reached'}\n\n")
        f.write(f"Initial Best:     {initial_results[0]['human_prob']*100:.2f}% Human (detector caught it)\n")
        f.write(f"Final Best:       {final_best_enhanced['human_prob']*100:.2f}% Human (detector {'fooled!' if final_best_enhanced['human_prob'] > 0.5 else 'caught it'})\n")
        f.write(f"Improvement:      {(final_best_enhanced['human_prob'] - initial_results[0]['human_prob'])*100:+.2f}%\n")
        f.write(f"Generations:      {len(history_enhanced)}/{NUM_GENERATIONS}\n")
        f.write(f"Runtime:          {elapsed_enhanced}\n\n")

        f.write("=" * 80 + "\n")
        f.write("MUTATION STRATEGY EFFECTIVENESS\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"{'Rank':<6}{'Strategy':<35}{'Success Rate':<15}{'Attempts':<12}{'Avg Δ':<12}\n")
        f.write("-" * 80 + "\n")
        for i, (_, row) in enumerate(mutation_stats_enhanced.iterrows(), 1):
            f.write(f"{i:<6}{row['strategy']:<35}{row['success_rate']*100:>6.1f}%       {row['attempts']:>6.0f}      {row['avg_improvement']:>+8.4f}\n")

        f.write("\n" + "=" * 80 + "\n")
        f.write("VICTORIAN MARKER EVOLUTION\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"{'Marker':<30}{'Initial':<10}{'Final':<10}{'Change':<10}\n")
        f.write("-" * 80 + "\n")
        f.write(f"{'Archaic conjunctions':<30}{initial_markers['archaic_conj']:<10}{final_markers['archaic_conj']:<10}{final_markers['archaic_conj'] - initial_markers['archaic_conj']:+d}\n")
        f.write(f"{'Past tense markers':<30}{initial_markers['past_tense']:<10}{final_markers['past_tense']:<10}{final_markers['past_tense'] - initial_markers['past_tense']:+d}\n")
        f.write(f"{'First-person pronouns':<30}{initial_markers['first_person']:<10}{final_markers['first_person']:<10}{final_markers['first_person'] - initial_markers['first_person']:+d}\n")
        f.write(f"{'Modern transitions':<30}{initial_markers['modern_trans']:<10}{final_markers['modern_trans']:<10}{final_markers['modern_trans'] - initial_markers['modern_trans']:+d}\n")
        f.write(f"{'Victorian vocabulary':<30}{initial_markers['victorian_vocab']:<10}{final_markers['victorian_vocab']:<10}{final_markers['victorian_vocab'] - initial_markers['victorian_vocab']:+d}\n")
        f.write(f"{'Semicolons':<30}{initial_markers['semicolons']:<10}{final_markers['semicolons']:<10}{final_markers['semicolons'] - initial_markers['semicolons']:+d}\n")
        f.write(f"{'Em-dashes':<30}{initial_markers['em_dashes']:<10}{final_markers['em_dashes']:<10}{final_markers['em_dashes'] - initial_markers['em_dashes']:+d}\n")

        f.write("\n" + "=" * 80 + "\n")
        f.write("SCIENTIFIC INTERPRETATION\n")
        f.write("=" * 80 + "\n\n")

        if final_best_enhanced['human_prob'] >= TARGET_FITNESS:
            f.write("✅ DETECTOR IS VULNERABLE TO EVOLUTIONARY ATTACKS\n\n")
            f.write("Key Findings:\n")
            f.write(f"  • GA successfully evolved text from {initial_results[0]['human_prob']*100:.2f}% to {final_best_enhanced['human_prob']*100:.2f}% Human\n")
            f.write(f"  • Victorian markers emerged through guided mutation\n")
            f.write(f"  • Most effective strategy: {mutation_stats_enhanced.iloc[0]['strategy']} ({mutation_stats_enhanced.iloc[0]['success_rate']*100:.1f}% success)\n")
            f.write(f"  • Evolution discovered that {mutation_stats_enhanced.iloc[0]['strategy']} patterns fool detector\n\n")
            f.write("Implications:\n")
            f.write("  • Detector NOT production-ready without adversarial hardening\n")
            f.write("  • Evolutionary algorithms can systematically probe weaknesses\n")
            f.write("  • Victorian patterns are learnable through guided optimization\n")
            f.write("  • Recommend: Adversarial training with GA-evolved examples\n\n")
            f.write("Next Steps:\n")
            f.write("  1. Retrain detector with adversarial examples from this GA\n")
            f.write("  2. Implement adversarial training loop\n")
            f.write("  3. Test robustness with multiple GA runs\n")
            f.write("  4. Red-team testing with human adversaries\n")
        else:
            f.write("✅ DETECTOR IS ROBUST AGAINST EVOLUTIONARY ATTACKS\n\n")
            f.write("Key Findings:\n")
            f.write(f"  • GA improved fitness from {initial_results[0]['human_prob']*100:.2f}% to {final_best_enhanced['human_prob']*100:.2f}% Human\n")
            f.write(f"  • Improvement of {(final_best_enhanced['human_prob'] - initial_results[0]['human_prob'])*100:+.2f}% but below {TARGET_FITNESS*100:.0f}% target\n")
            f.write(f"  • Even enhanced mutations could not fool detector\n")
            f.write(f"  • Deep structural features appear unfakeable\n\n")
            f.write("Implications:\n")
            f.write("  • Detector learned deep patterns beyond superficial markers\n")
            f.write("  • Victorian style cannot be easily replicated by AI\n")
            f.write("  • Domain-specific training provides robustness\n")
            f.write("  • Detector MAY be suitable for production deployment\n\n")
            f.write("Next Steps:\n")
            f.write("  1. Additional red-team testing recommended\n")
            f.write("  2. Test with human-crafted adversarial examples\n")
            f.write("  3. Validate on diverse Victorian authors\n")
            f.write("  4. Monitor for concept drift over time\n")

        f.write("\n" + "=" * 80 + "\n")
        f.write("FILES GENERATED\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"  • evolution_history_enhanced.csv\n")
        f.write(f"  • mutation_strategy_effectiveness_enhanced.csv\n")
        f.write(f"  • victorian_markers_evolution_enhanced.csv\n")
        f.write(f"  • final_population_enhanced.csv\n")
        f.write(f"  • best_evolved_text_enhanced.txt\n")
        f.write(f"  • original_vs_enhanced_comparison.csv\n")
        f.write(f"  • victorian_markers_evolution_enhanced.png\n")
        f.write(f"  • EXPERIMENT_SUMMARY_ENHANCED.txt (this file)\n")

        f.write("\n" + "=" * 80 + "\n")
        f.write("END OF REPORT\n")
        f.write("=" * 80 + "\n")

    print(f"✅ Saved: {OUTPUT_DIR}/EXPERIMENT_SUMMARY_ENHANCED.txt")

    print("\n" + "=" * 80)
    print("✅ ALL ENHANCED RESULTS SAVED SUCCESSFULLY!")
    print("=" * 80)
    print(f"\n📁 Output directory: {OUTPUT_DIR}/")
    print("\n📊 Files created:")
    print("   1. evolution_history_enhanced.csv - Generation-by-generation fitness")
    print("   2. mutation_strategy_effectiveness_enhanced.csv - Which strategies worked")
    print("   3. victorian_markers_evolution_enhanced.csv - Marker emergence data")
    print("   4. final_population_enhanced.csv - All final individuals ranked")
    print("   5. best_evolved_text_enhanced.txt - Winner with full analysis")
    print("   6. original_vs_enhanced_comparison.csv - Original vs Enhanced GA")
    print("   7. victorian_markers_evolution_enhanced.png - Evolution visualization")
    print("   8. EXPERIMENT_SUMMARY_ENHANCED.txt - Complete summary report")
    print("\n🎉 Your enhanced GA achieved " +
          f"{final_best_enhanced['human_prob']*100:.2f}% Human confidence!")
    print(f"   Target was {TARGET_FITNESS*100:.0f}% - " +
          ("✅ SUCCESS!" if final_best_enhanced['human_prob'] >= TARGET_FITNESS else "❌ Not reached"))
    print("=" * 80)

💾 SAVING COMPLETE ENHANCED GA RESULTS

📊 ENHANCED GA FINAL STATISTICS:
Success: ✅ YES
Initial best:  0.06% Human
Final best:    97.61% Human
Improvement:   +97.55%
Generations:   6
Runtime:       0:10:33.445836
Target:        85% Human

✅ Saved: /content/drive/MyDrive/precog/task4_outputs/evolution_history_enhanced.csv
✅ Saved: /content/drive/MyDrive/precog/task4_outputs/mutation_strategy_effectiveness_enhanced.csv
✅ Saved: /content/drive/MyDrive/precog/task4_outputs/victorian_markers_evolution_enhanced.csv
✅ Saved: /content/drive/MyDrive/precog/task4_outputs/final_population_enhanced.csv
✅ Saved: /content/drive/MyDrive/precog/task4_outputs/best_evolved_text_enhanced.txt
✅ Saved: /content/drive/MyDrive/precog/task4_outputs/original_vs_enhanced_comparison.csv

📊 Creating Victorian marker evolution visualization...
✅ Saved: /content/drive/MyDrive/precog/task4_outputs/victorian_markers_evolution_enhanced.png
✅ Saved: /content/drive/MyDrive/precog/task4_outputs/EXPERIMENT_SUMMARY_ENHANCED.